[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_computing/05_numerical_stability_in_deep_learning/first_principles.ipynb)

# Topic 05: Numerical Stability in Deep Learning

## 1. First-Principles Intuition & Motivation

Every earlier topic in this module assumed binary64: $16$ decimal digits, an exponent window of $10^{\pm 308}$, and a unit roundoff so small that most bounds looked academic. Deep learning discards all of that. A modern training step multiplies bf16 matrices ($u = 2^{-8}$, about $2.4$ decimal digits), accumulates into fp32, stores master weights in fp32, communicates gradients in fp16 or fp8, and computes softmax over logits that can legitimately reach $\pm 30$.

In that regime the theorems stop being academic and become the operating manual. Three numbers explain most training failures ever reported:

- $\log(65504) = 11.09$ — the largest logit whose exponential survives in fp16.
- $2^{-24} \approx 6 \times 10^{-8}$ — the smallest positive fp16 subnormal; any gradient below it becomes exactly zero.
- $2^{-11} \approx 4.9 \times 10^{-4}$ — fp16's unit roundoff; a weight update smaller than this *relative to the weight* is absorbed and has no effect at all.

The rest of the notebook derives the standard countermeasures — max-subtraction, fused losses, loss scaling, fp32 master weights and accumulators, careful normalization statistics, correct $\epsilon$ placement — each as a consequence of Topics 01–03 rather than as a piece of empirical lore.

### The two windows

A floating-point format is a pair of windows, and every stability rule keeps a quantity inside one of them.

$$
\underbrace{\left[ 2^{e_{\min}}, \; 2^{e_{\max}+1} \right)}_{\textbf{range window (exponent bits)}} \qquad \text{and} \qquad \underbrace{u = 2^{-p}}_{\textbf{precision window (significand bits)}}
$$

- **Range failures** are loud: `inf`, `NaN`, a loss that becomes `nan` at step 400 and stays there. Causes: $e^{z}$ with large $z$, squared norms of large activations, products of many quantities greater than one, division by an underflowed denominator.
- **Precision failures** are silent: a metric that freezes, a variance estimate that is pure noise, a weight that never moves, a gradient that vanished into a subnormal. Causes: cancellation, absorption of small addends into large accumulators, and reductions longer than $1/u$ terms.

The silent class is far more dangerous, because training continues and produces a plausible-looking curve. **The diagnostic habit this module teaches is to compute, for each intermediate, the largest and smallest magnitude it can attain and compare them against the format's two windows — before running anything.**

An illustrative check, worth internalizing:

```python
# What is the largest logit fp16 can exponentiate?
#   float16 max = 65504,  log(65504) = 11.0904...
# Any z_j - max(z) is <= 0, so exp is in (0, 1]: the max-subtraction
# trick converts an unbounded range problem into a bounded one.
```

## 2. Rigorous Mathematical Definitions & Theorem Statements

### Definition 2.1 (Low-precision formats)

A binary floating-point format with $p$ significand bits (including the implicit leading bit) and exponent field of $w$ bits represents

$$
x = \pm\, m \times 2^{e}, \qquad m \in [1, 2), \quad e \in [e_{\min}, e_{\max}], \quad u = 2^{-p}
$$

The formats that matter in deep learning:

| Format | Sign/Exp/Mantissa | $p$ | Min normal | Max | $u = 2^{-p}$ | $\log(\text{max})$ |
|---|---|---|---|---|---|---|
| fp64 | 1/11/52 | 53 | $2.2 \times 10^{-308}$ | $1.8 \times 10^{308}$ | $1.1 \times 10^{-16}$ | $709.8$ |
| fp32 | 1/8/23 | 24 | $1.2 \times 10^{-38}$ | $3.4 \times 10^{38}$ | $6.0 \times 10^{-8}$ | $88.7$ |
| tf32 | 1/8/10 | 11 | $1.2 \times 10^{-38}$ | $3.4 \times 10^{38}$ | $4.9 \times 10^{-4}$ | $88.7$ |
| bf16 | 1/8/7 | 8 | $1.2 \times 10^{-38}$ | $3.4 \times 10^{38}$ | $3.9 \times 10^{-3}$ | $88.7$ |
| fp16 | 1/5/10 | 11 | $6.1 \times 10^{-5}$ | $65504$ | $4.9 \times 10^{-4}$ | $11.09$ |
| fp8 E4M3 | 1/4/3 | 4 | $2.0 \times 10^{-3}$ | $448$ | $6.3 \times 10^{-2}$ | $6.10$ |
| fp8 E5M2 | 1/5/2 | 3 | $6.1 \times 10^{-5}$ | $57344$ | $1.3 \times 10^{-1}$ | $10.96$ |

**The bf16/fp16 dichotomy.** Both occupy 16 bits; they spend them differently. bf16 keeps fp32's exponent field, so *any* value representable in fp32 is representable in bf16 (with more rounding) — casting never overflows or underflows, and **no loss scaling is needed**. fp16 spends 3 more bits on the significand, buying $8\times$ the precision inside a range window of only $[6 \times 10^{-5}, 65504]$ — hence loss scaling, overflow guards, and dynamic-range engineering.

**Subnormals.** Below $2^{e_{\min}}$ the formats degrade gracefully to subnormals with reduced precision, down to $2^{e_{\min}-p+1}$: for fp16 that is $2^{-24} \approx 5.96 \times 10^{-8}$. On some hardware subnormals are flushed to zero (FTZ/DAZ), which turns a graceful degradation into a hard cliff — worth verifying, because it moves the underflow threshold up by a factor of $1024$.

### Theorem 2.2 (Log-sum-exp: identity, stability, and error bound)

**Identity.** For any $z \in \mathbb{R}^{n}$ and any $m \in \mathbb{R}$,

$$
\log \sum_{j=1}^{n} e^{z_j} = m + \log \sum_{j=1}^{n} e^{z_j - m}
$$

exactly, in real arithmetic (factor $e^{m}$ out of the sum). Choosing $m = \max_j z_j$ gives the **stable form**.

**Stability.** With $m = \max_j z_j$: every exponent satisfies $z_j - m \le 0$, so $e^{z_j - m} \in (0, 1]$ — **overflow is impossible**. At least one term equals exactly $1$, so $\sum_j e^{z_j - m} \in [1, n]$ — **the sum cannot underflow to zero**, and the subsequent $\log$ has argument in $[1, n]$, where it is well conditioned.

**Error bound.** Let $\hat{L}$ denote the computed value of $L = \log\sum_j e^{z_j}$ by the stable algorithm, in a format with unit roundoff $u$, assuming `exp` and `log` are faithful to $1$ ulp. Then

$$
\left\vert \hat{L} - L \right\vert \; \le \; \left( n + 3 \right) u \, \vert L \vert + \left( n + 3 \right) u + O(u^{2})
$$

i.e. the *absolute* error is $O(nu)$ — and because $L \ge m$, the relative error is $O(nu)$ as well whenever $\vert m \vert \gtrsim 1$. The naive algorithm has *no* error bound at all: it returns `inf` for $m \gt \log(\text{max})$ and $-$`inf` for $m \lt \log(\text{min})$.

**Corollary (softmax).**

$$
\mathrm{softmax}(z)_i = \frac{e^{z_i - m}}{\sum_j e^{z_j - m}}
$$

with numerator in $(0, 1]$ and denominator in $[1, n]$: every intermediate is bounded, and the result has relative error $O(nu)$.

### Definition 2.3 (Losses computed from logits)

**Softmax cross-entropy.** For logits $z \in \mathbb{R}^{n}$ and true class $y$,

$$
\ell(z, y) = -\log \mathrm{softmax}(z)_y = -z_y + \log\sum_{j} e^{z_j} = -z_y + m + \log\sum_j e^{z_j - m}
$$

The right-hand form never constructs the probability $p_y$, so it cannot underflow it to zero, and it has the exact gradient

$$
\frac{\partial \ell}{\partial z_i} = \mathrm{softmax}(z)_i - \mathbb{1}[i = y]
$$

which is bounded in $[-1, 1]$ — one of the reasons the fused kernel is also better behaved in the backward pass.

**Binary cross-entropy from logits.** For a single logit $z$ and label $y \in \{0, 1\}$,

$$
\ell(z, y) = \max(z, 0) - yz + \log\!\left( 1 + e^{-\vert z \vert} \right)
$$

Every term is bounded: $e^{-\vert z \vert} \in (0, 1]$, so `log1p` receives an argument in $(1, 2]$ where it is exact to $O(u)$. This is `BCEWithLogitsLoss`; the unfused $-[y\log\sigma(z) + (1-y)\log(1-\sigma(z))]$ returns `inf` for $\vert z \vert \gtrsim 17$ in fp32 and $\vert z \vert \gtrsim 8$ in fp16.

**Rule.** *Never let a probability be an intermediate.* Losses take logits; samplers take logits; comparisons take log-probabilities. The probability is an output for humans, not a quantity to compute with.

### Theorem 2.4 (Loss scaling is exact)

Let $\mathcal{L}$ be the loss and $S \gt 0$ a constant. By linearity of differentiation,

$$
\nabla_\theta \left( S \cdot \mathcal{L} \right) = S \, \nabla_\theta \mathcal{L}
$$

so computing the backward pass on $S\mathcal{L}$ and dividing the resulting gradients by $S$ before the optimizer step recovers $\nabla_\theta\mathcal{L}$ **exactly in real arithmetic**, and to within one rounding in floating point.

**Why it matters.** In fp16 the smallest representable positive value is $2^{-24}$. Measured gradient histograms in real networks place a substantial mass of components below $2^{-24}$ — those components become exactly zero, and the corresponding weights never update. Multiplying by $S = 2^{k}$ shifts the entire histogram $k$ binades upward, into the representable window, at the cost of moving the top of the histogram $k$ binades closer to the overflow threshold $65504$.

**Choosing $S$.** The constraint is a two-sided window on the gradient magnitudes $g$:

$$
2^{-24} \; \lt \; S \, g_{\min} \qquad \text{and} \qquad S \, g_{\max} \; \lt \; 65504
$$

which is feasible exactly when the gradient dynamic range $g_{\max}/g_{\min}$ is below fp16's total range $65504 \cdot 2^{24} \approx 1.1 \times 10^{12}$.

**Dynamic loss scaling** (the practical algorithm): start at $S = 2^{16}$; if any gradient is non-finite, **skip the step** and set $S \leftarrow S/2$; if $N$ consecutive steps succeed (typically $N = 2000$), set $S \leftarrow 2S$. This tracks the moving histogram without tuning, and the skipped steps cost a negligible fraction of training.

**bf16 needs none of this**: its exponent range equals fp32's, so no gradient that was representable in fp32 underflows in bf16.

### Definition 2.5 (Mixed-precision training)

The standard recipe (Micikevicius et al., 2018) maintains **two copies** of the parameters:

$$
\theta^{(32)} \in \text{fp32 (master)}, \qquad \theta^{(16)} = \mathrm{fl}_{16}\!\left( \theta^{(32)} \right) \in \text{fp16/bf16 (compute)}
$$

and executes, per step:

1. Cast $\theta^{(32)} \to \theta^{(16)}$.
2. Forward pass in 16-bit; matrix products **accumulate in fp32** inside the tensor cores.
3. Multiply the loss by $S$; backward pass in 16-bit.
4. Cast gradients to fp32 and divide by $S$; check finiteness (skip the step if not).
5. Optionally clip; update $\theta^{(32)}$ with the optimizer, whose state ($m$, $v$) also lives in fp32.

**Why the master copy is not optional.** A weight update is absorbed — has literally no effect — when it falls below half an ulp of the weight:

$$
\frac{\eta \vert g \vert}{\vert \theta \vert} \; \lt \; \frac{u}{2}
$$

For fp16, $u/2 = 2^{-12} \approx 2.4 \times 10^{-4}$. Typical late-training relative updates are $10^{-4}$–$10^{-6}$: **the majority of updates would be silently discarded**. The fp32 master copy has $u/2 = 3 \times 10^{-8}$, giving four extra decades of headroom, and the accumulated small updates eventually cross the fp16 grid on the next cast.

**Why accumulation must be wide.** A reduction over $n$ terms in a format with unit roundoff $u$ stalls when $n \gtrsim 1/u$ (Topic 02): $1/u = 2048$ for fp16, $256$ for bf16, $1.7\times10^{7}$ for fp32. Since a single $H = 4096$ dot product already has $n = 4096$, accumulating in 16-bit is not an option — which is exactly why tensor cores accumulate in fp32 by hardware design.

### Theorem 2.6 (Vanishing and exploding gradients as a conditioning statement)

For a depth-$L$ network with layer maps $h_{k} = f_k(h_{k-1})$, the backpropagated gradient is a product of Jacobians:

$$
\frac{\partial \mathcal{L}}{\partial h_0} = \frac{\partial \mathcal{L}}{\partial h_L} \prod_{k=L}^{1} J_k, \qquad J_k = \frac{\partial h_k}{\partial h_{k-1}}
$$

Hence the submultiplicative bound and its matching lower bound

$$
\left\Vert \frac{\partial \mathcal{L}}{\partial h_0} \right\Vert_2 \le \left\Vert \frac{\partial \mathcal{L}}{\partial h_L} \right\Vert_2 \prod_{k=1}^{L} \sigma_{\max}(J_k), \qquad \ge \left\Vert \frac{\partial \mathcal{L}}{\partial h_L} \right\Vert_2 \prod_{k=1}^{L} \sigma_{\min}(J_k)
$$

If $\sigma_{\max}(J_k) \le \gamma$ for all $k$, the gradient decays at worst like $\gamma^{L}$; if $\sigma_{\min}(J_k) \ge \Gamma \gt 1$, it grows at least like $\Gamma^{L}$. Both are exponential in depth, and the *condition number of the composition* is bounded by $\prod_k \kappa(J_k)$ — the product form of Topic 03's $\kappa(AB) \le \kappa(A)\kappa(B)$.

**Numerical consequences by format.** With $\gamma = 0.9$ and $L = 100$: $\gamma^{L} = 2.7 \times 10^{-5}$ — still normal in fp32, but already at fp16's minimum normal $6.1 \times 10^{-5}$. With $\gamma = 0.8$, $L = 100$: $2 \times 10^{-10}$, flushed to zero in fp16 and merely small in fp32.

**Countermeasures, each a spectral statement.**

- **Variance-preserving initialization** (Glorot, He): choose the weight scale so that $\mathbb{E}[\sigma^{2}(J_k)] \approx 1$, i.e. $\mathrm{Var}(W) = 2/n_{\text{in}}$ for ReLU.
- **Residual connections**: $J_k = I + \tilde{J}_k$ has singular values clustered near $1$, so the product does not decay; the gradient has an $O(1)$ path to every layer.
- **Normalization layers**: rescale activations so the Jacobian's scale is data-independent, and project out the component along the mean/scale direction.
- **Orthogonal / unitary parameterizations** (RNNs): enforce $\sigma_i(J) = 1$ exactly.
- **Gradient clipping**: $g \leftarrow g \cdot \min(1, c/\Vert g \Vert)$ — a hard cap on the product's growth, and the standard defence in RNNs and transformers alike.

### Definition 2.7 (Normalization statistics and Adam's $\epsilon$)

**Normalization.** BatchNorm/LayerNorm compute

$$
\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^{2} + \epsilon}}, \qquad \mu = \frac{1}{n}\sum_i x_i, \quad \sigma^{2} = \frac{1}{n}\sum_i (x_i - \mu)^{2}
$$

Two numerical requirements: (i) $\sigma^{2}$ must be computed by a two-pass or Welford recurrence, never as $\overline{x^{2}} - \mu^{2}$, whose relative error is amplified by $\mu^{2}/\sigma^{2}$ (Topic 02, Derivation 3.7); (ii) $\epsilon$ sits **inside** the square root, so the denominator is bounded below by $\sqrt{\epsilon}$ and the derivative $\frac{\partial}{\partial \sigma^{2}}(\sigma^{2}+\epsilon)^{-1/2}$ stays finite as $\sigma \to 0$.

**Adam.** With $\hat{m}_t, \hat{v}_t$ the bias-corrected moments, the update is

$$
\theta_{t+1} = \theta_t - \eta \, \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

Reading the fraction as a diagonal preconditioner $P = \mathrm{diag}\!\left( (\sqrt{\hat{v}_t} + \epsilon)^{-1} \right)$, its condition number is

$$
\kappa(P) = \frac{\sqrt{v_{\max}} + \epsilon}{\sqrt{v_{\min}} + \epsilon} \; \le \; \frac{\sqrt{v_{\max}} + \epsilon}{\epsilon}
$$

so **$\epsilon$ is a conditioning cap**, exactly the ridge parameter of Topic 03's Derivation 3.6, not a division guard. Two placements exist and are not equivalent:

$$
\frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon} \quad \text{(standard, Kingma \& Ba)} \qquad \text{versus} \qquad \frac{\hat{m}}{\sqrt{\hat{v} + \epsilon}} \quad \text{(used by some frameworks)}
$$

The second caps the step at $\hat{m}/\sqrt{\epsilon}$ (a bound on the *squared* gradient scale) and is scale-covariant in a different way; the first caps it at $\hat{m}/\epsilon$. Both matter in low precision: with $\epsilon = 10^{-8}$ stored in fp16, $\epsilon$ **rounds to zero** (fp16's minimum subnormal is $6 \times 10^{-8}$), removing the cap entirely — which is why optimizer state is kept in fp32.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Derivation 3.1: Overflow thresholds, exactly

**Question.** For which logits does the naive $\sum_j e^{z_j}$ overflow?

**Answer.** $e^{z}$ exceeds the format maximum $X_{\max}$ when $z \gt \log X_{\max}$:

| Format | $X_{\max}$ | $\log X_{\max}$ | Naive softmax overflows at |
|---|---|---|---|
| fp64 | $1.8 \times 10^{308}$ | $709.78$ | $z \gt 709.78$ |
| fp32 | $3.4 \times 10^{38}$ | $88.72$ | $z \gt 88.72$ |
| bf16 | $3.4 \times 10^{38}$ | $88.72$ | $z \gt 88.72$ |
| fp16 | $65504$ | $11.09$ | $z \gt 11.09$ |
| fp8 E4M3 | $448$ | $6.10$ | $z \gt 6.10$ |

**Why $11.09$ is a *small* number.** Logits are pre-softmax scores; a confidently trained classifier routinely produces $z \in [-20, 20]$, and an unnormalized attention score $q^{\top}k$ over $d = 64$ dimensions with unit-variance entries has standard deviation $\sqrt{d} = 8$ — which is precisely why attention divides by $\sqrt{d_k}$. In fp16, therefore, **naive softmax overflows in ordinary operation**, not in edge cases.

**The underflow side.** $e^{z}$ flushes to zero when $z \lt \log X_{\min}$: $-103.28$ for fp32 (subnormal limit), $-17.33$ for fp16. If *all* logits underflow, the denominator is $0$ and the result is `0/0 = NaN`. The naive algorithm therefore fails on both ends, with a usable window of only $\approx 28$ units in fp16.

**After max-subtraction.** Every argument is in $(-\infty, 0]$ and every term in $(0, 1]$, with the maximum term exactly $1$. The denominator is in $[1, n]$. **Both failure modes are removed unconditionally, for all inputs, in all formats.**

$$
\text{usable logit range: naive fp16} \approx 28, \qquad \text{stable: unbounded}
$$

### Derivation 3.2: Proof of the log-sum-exp identity and its error bound

**Identity.** Factor $e^{m}$ from every term:

$$
\sum_{j} e^{z_j} = e^{m}\sum_{j} e^{z_j - m} \quad \Longrightarrow \quad \log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m} \qquad \blacksquare
$$

valid for every $m$, so the identity is an exact algebraic rewrite — the choice $m = \max_j z_j$ is made purely for numerical reasons, which is the recurring pattern of Topic 02.

**Error analysis.** Write $t_j = z_j - m \le 0$ and $\Sigma = \sum_j e^{t_j} \in [1, n]$.

**Step 1 — subtraction.** $\mathrm{fl}(z_j - m) = (z_j - m)(1 + \delta_j)$, $\vert\delta_j\vert \le u$. Since $e^{t(1+\delta)} = e^{t}e^{t\delta} = e^{t}(1 + t\delta + O(\delta^{2}))$, the perturbation of each term is relative of size $\vert t_j \vert u$. Note this grows with the *spread* of the logits: for $\vert t_j \vert \le T$ the relative term error is $\le Tu$.

**Step 2 — exponentials.** A faithful `exp` contributes another relative $u$: computed term $= e^{t_j}(1 + \eta_j)$ with $\vert \eta_j \vert \le (T + 1)u + O(u^{2})$.

**Step 3 — summation.** Recursive summation of $n$ nonnegative terms gives (Topic 02, Theorem 2.3) $\hat{\Sigma} = \Sigma(1 + \theta)$ with $\vert\theta\vert \le \gamma_{n-1} + \max_j\vert\eta_j\vert \approx (n - 1 + T + 1)u$. Crucially, **all terms are positive, so there is no cancellation** and the condition number of this summation is exactly $1$.

**Step 4 — logarithm.** $\log$ converts relative input error to absolute output error: $\log(\Sigma(1+\theta)) = \log\Sigma + \theta + O(\theta^{2})$, plus the library's own $u$. Adding $m$ (one more rounding, relative $u$ on a quantity of size $\vert m \vert$):

$$
\left\vert \hat{L} - L \right\vert \; \le \; \underbrace{(n + T + 1)u}_{\text{from } \Sigma} + \underbrace{u\vert\log\Sigma\vert}_{\log} + \underbrace{u\vert m \vert}_{\text{final add}} + O(u^{2})
$$

Since $L = m + \log\Sigma$ and $\log\Sigma \in [0, \log n]$, this is $O\!\left( (n + T)u \right)$ absolute — bounded, computable, and independent of any overflow risk. $\blacksquare$

**Reading.** The bound degrades with the *spread* $T$ of the logits, not with their magnitude: shifting all logits by $10^{6}$ costs nothing after max-subtraction, while spreading them over $10^{6}$ costs $6$ digits. This is why logit *scaling* (temperature, $1/\sqrt{d_k}$, logit soft-capping) is a numerical intervention as much as a modelling one.

### Derivation 3.3: The fused cross-entropy losses

**Softmax cross-entropy.** Substitute the softmax definition and let the logs cancel *on paper*:

$$
\ell = -\log\frac{e^{z_y}}{\sum_j e^{z_j}} = -z_y + \log\sum_j e^{z_j} = -z_y + m + \log\sum_j e^{z_j - m}
$$

**Why the unfused version fails.** Computing $p_y$ first commits two errors: (i) if $z_y - m \lt \log(X_{\min})$ then $p_y$ underflows to $0$ and $\log 0 = -\infty$ — an infinite loss from a merely *very wrong* prediction; (ii) even without underflow, $\log$ near $0$ has condition number

$$
\kappa_{\log}(p) = \left\vert \frac{p \cdot (1/p)}{\log p} \right\vert = \frac{1}{\vert \log p \vert}
$$

which is benign for small $p$ but *diverges as $p \to 1$* — the well-classified case, where the loss is a tiny number obtained as $\log$ of something indistinguishable from $1$. The fused form computes $\log\sum_j e^{z_j - m} \in [0, \log n]$ directly, so both regimes are handled.

**Gradient.** Differentiating the fused expression,

$$
\frac{\partial \ell}{\partial z_i} = -\mathbb{1}[i = y] + \frac{e^{z_i - m}}{\sum_j e^{z_j - m}} = p_i - \mathbb{1}[i = y] \; \in \; [-1, 1]
$$

Bounded gradients regardless of how wrong the prediction is — a *structural* stability property, and the reason the fused kernel is safe in fp16 where an unfused $\frac{1}{p_y}\cdot\frac{\partial p_y}{\partial z}$ would overflow.

**Binary cross-entropy from logits.** Start from $\sigma(z) = 1/(1+e^{-z})$:

$$
\ell = -y\log\sigma(z) - (1-y)\log(1 - \sigma(z))
$$

Use $\log\sigma(z) = -\log(1 + e^{-z})$ and $\log(1 - \sigma(z)) = -z - \log(1 + e^{-z})$:

$$
\ell = y\log(1 + e^{-z}) + (1-y)\left( z + \log(1 + e^{-z}) \right) = (1 - y)z + \log(1 + e^{-z})
$$

This is stable for $z \ge 0$ but overflows $e^{-z}$ for large negative $z$. Symmetrize by pulling out $\max(z, 0)$:

$$
\boxed{\ell = \max(z, 0) - yz + \mathrm{log1p}\!\left( e^{-\vert z \vert} \right)}
$$

Now $e^{-\vert z \vert} \in (0, 1]$ always, `log1p` receives an argument in $(1, 2]$ where it is accurate to $O(u)$ (Topic 02, Derivation 3.6), and no term can overflow. For $z = 40$, $y = 0$ the value is $40 + \mathrm{log1p}(4.2\times10^{-18}) = 40$ to full precision, whereas the unfused form returns `inf`.

### Derivation 3.4: Loss scaling and master weights, quantitatively

**Part 1 — the underflow problem.** Let the gradient components have magnitudes spread over several decades, with a lower tail below fp16's smallest subnormal $2^{-24} = 5.96 \times 10^{-8}$. Those components round to exactly zero:

$$
\mathrm{fl}_{16}(g) = 0 \iff \vert g \vert \lt 2^{-25} \; (\text{round-to-nearest at the subnormal boundary})
$$

and the corresponding weights receive no update, permanently, for as long as the gradient stays that small.

**Part 2 — the fix and its exactness.** Backpropagating $S\mathcal{L}$ yields $S\nabla\mathcal{L}$ by linearity; dividing by $S$ in fp32 before the update restores the true gradient. Because $S = 2^{k}$ is a power of two, both the multiplication and the division are **exact** (they only modify the exponent field), so loss scaling introduces *zero* additional rounding error — a rare free lunch.

**Part 3 — choosing $S$.** Requiring the whole histogram inside the window:

$$
S \gt \frac{2^{-24}}{g_{\min}} \qquad \text{and} \qquad S \lt \frac{65504}{g_{\max}}
$$

**Worked example.** $g_{\max} = 2^{-6}$, $g_{\min} = 2^{-32}$. Lower constraint: $S \gt 2^{-24}/2^{-32} = 2^{8}$. Upper: $S \lt 65504 \cdot 2^{6} \approx 2^{22}$. Any $S \in (2^{8}, 2^{22})$ works; the standard initial choice $S = 2^{16}$ sits comfortably in the middle. Infeasibility occurs only when $g_{\max}/g_{\min} \gt 65504 \cdot 2^{24} \approx 2^{40}$.

**Part 4 — dynamic scaling.** The histogram moves during training, so fix $S$ adaptively:

```text
S = 2**16
for each step:
    g = backward(S * loss)
    if any(not isfinite(g)):      # overflow detected
        S = S / 2;  skip the update
    else:
        update(theta, g / S)
        if steps_since_overflow >= 2000:  S = 2 * S
```

The overflow check is a reduction over gradients — cheap — and skipped steps are a few per thousand.

**Part 5 — master weights.** A weight update is absorbed when

$$
\eta\vert g \vert \; \lt \; \tfrac{1}{2}\mathrm{ulp}(\theta) = \tfrac{1}{2}u\vert\theta\vert \quad \Longleftrightarrow \quad \frac{\eta\vert g\vert}{\vert\theta\vert} \lt \frac{u}{2}
$$

| Format | $u/2$ | Relative updates that are lost |
|---|---|---|
| fp16 | $2.4 \times 10^{-4}$ | most of late training |
| bf16 | $2.0 \times 10^{-3}$ | even more |
| fp32 | $3.0 \times 10^{-8}$ | essentially none |

With a typical relative update of $10^{-5}$, fp16 weights would ignore **every** step after the initial transient while the loss curve flattens deceptively. Keeping $\theta^{(32)}$ in fp32 lets sub-ulp updates accumulate until they collectively cross the fp16 grid — the same "track residuals, not absolutes" principle as Kahan summation and Welford's variance.

### Derivation 3.5: Normalization-layer numerics

**The variance trap.** With activations of mean $\mu$ and standard deviation $\sigma$, the one-pass identity $\sigma^{2} = \overline{x^{2}} - \mu^{2}$ subtracts two quantities of size $\mu^{2}$ to obtain one of size $\sigma^{2}$. By Topic 02's cancellation bound the relative error is amplified by

$$
\frac{\mu^{2} + \overline{x^{2}}}{\sigma^{2}} \approx \frac{2\mu^{2}}{\sigma^{2}}
$$

**Numbers.** For activations with $\mu = 50$, $\sigma = 0.1$ in fp32 ($u = 6 \times 10^{-8}$): amplification $5 \times 10^{5}$, giving relative error $\gtrsim 3\%$ on $\sigma^{2}$ before summation error is even counted. In bf16 ($u = 3.9\times10^{-3}$) the same amplification returns error $\gg 1$: the variance is pure noise, and can be negative. Hence:

- Compute statistics in **fp32** even when activations are 16-bit (all frameworks do this).
- Use the **two-pass** or **Welford** recurrence, which subtracts only residuals $x_i - m$ of size $\sigma$.
- For distributed BatchNorm, combine per-device $(n, \mu, M_2)$ triples with the Chan–Golub–LeVeque parallel merge, never by exchanging raw $\sum x$ and $\sum x^{2}$.

**Why $\epsilon$ is inside the square root.** The normalized activation is $\hat{x} = (x - \mu)(\sigma^{2} + \epsilon)^{-1/2}$. Two properties follow:

1. **Bounded output**: $\vert \hat{x} \vert \le \vert x - \mu \vert/\sqrt{\epsilon}$, finite even for a constant batch ($\sigma = 0$), where the outside placement $\sigma + \epsilon$ would also be finite but the derivative would not be.
2. **Bounded derivative**: $\frac{\partial}{\partial\sigma^{2}}(\sigma^{2}+\epsilon)^{-1/2} = -\tfrac{1}{2}(\sigma^{2}+\epsilon)^{-3/2}$, bounded by $\tfrac{1}{2}\epsilon^{-3/2}$ — the backward pass cannot produce `inf` for degenerate batches. With $\epsilon$ *outside* ($\sigma + \epsilon$) the forward pass is fine but $\frac{\partial\sigma}{\partial\sigma^{2}} = \frac{1}{2\sigma}$ blows up as $\sigma \to 0$.

**Magnitude of $\epsilon$.** PyTorch defaults to $10^{-5}$ (BatchNorm) and $10^{-5}$ (LayerNorm); TensorFlow uses $10^{-3}$ for BatchNorm. In fp16 activations, $\epsilon = 10^{-5}$ is representable ($\gt 6\times10^{-8}$) but $\epsilon^{2}$ is not — one more reason the statistics path runs in fp32. The choice interacts with conditioning exactly as in Topic 03: $\epsilon$ caps the amplification of the normalization at $1/\sqrt{\epsilon}$.

**RMSNorm.** $\hat{x} = x / \sqrt{\overline{x^{2}} + \epsilon}$ skips the mean subtraction entirely. This *removes* the cancellation (there is no subtraction of large equal quantities) and is cheaper — part of why it has displaced LayerNorm in large language models.

### Derivation 3.6: Adam's $\epsilon$ is a conditioning cap

**Setup.** Adam's step in coordinate $i$ is

$$
\Delta\theta_i = -\eta\,\frac{\hat{m}_i}{\sqrt{\hat{v}_i} + \epsilon}
$$

Interpreting $P = \mathrm{diag}\!\left( (\sqrt{\hat{v}_i} + \epsilon)^{-1} \right)$ as a preconditioner (Topic 03, Sec. 4.2 step 4), its condition number is

$$
\kappa(P) = \frac{\max_i (\sqrt{\hat{v}_i} + \epsilon)}{\min_i (\sqrt{\hat{v}_i} + \epsilon)} \; \le \; \frac{\sqrt{v_{\max}} + \epsilon}{\epsilon} \; \approx \; \frac{\sqrt{v_{\max}}}{\epsilon}
$$

**Reading.** Without $\epsilon$, a coordinate whose gradient has been zero for many steps has $\hat{v}_i \to 0$ and receives an unbounded step — the preconditioner is singular. $\epsilon$ is precisely Tikhonov regularization of a diagonal preconditioner: it caps $\kappa(P)$, bounds the maximum step at $\eta/\epsilon$ times the momentum, and is chosen by the same "how many digits do I want?" logic as $\lambda$ in ridge regression.

**Three regimes as $\epsilon$ varies.**

| $\epsilon$ | Behaviour |
|---|---|
| $\epsilon \ll \sqrt{v_i}$ for all $i$ | Full sign-like normalization; steps of size $\approx \eta$ everywhere; maximal preconditioning, maximal noise sensitivity. |
| $\epsilon \sim \sqrt{v_i}$ | Interpolation: large-gradient coordinates normalized, small-gradient ones damped. |
| $\epsilon \gg \sqrt{v_i}$ | $\Delta\theta \approx -(\eta/\epsilon)\hat{m}$ — Adam degenerates to SGD with momentum and learning rate $\eta/\epsilon$. |

This is why $\epsilon$ is a *tuned* hyperparameter in practice (values from $10^{-8}$ to $10^{-4}$ across published recipes), not a constant: it selects a point on the SGD-to-signSGD spectrum.

**Placement.** $\frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon}$ caps the step at $\hat{m}/\epsilon$; $\frac{\hat{m}}{\sqrt{\hat{v} + \epsilon}}$ caps it at $\hat{m}/\sqrt{\epsilon}$. To make the two agree one must take $\epsilon_{\text{inside}} = \epsilon_{\text{outside}}^{2}$ — a factor that matters enormously: $\epsilon = 10^{-8}$ outside corresponds to $10^{-16}$ inside, which underflows in fp32 accumulation of $\hat{v}$. Porting a recipe between frameworks without squaring the constant silently changes the optimizer.

**Low precision.** fp16's smallest subnormal is $6 \times 10^{-8}$, so a default $\epsilon = 10^{-8}$ stored in fp16 becomes **exactly zero**, restoring the singular preconditioner. Similarly, $\hat{v}$ for small gradients ($g \sim 10^{-4}$, $g^{2} \sim 10^{-8}$) underflows in fp16. Both facts force the optimizer state into fp32 — which is also why optimizer state, not activations, dominates the memory footprint of large-model training, and why 8-bit optimizers use block-wise dynamic scaling to make it fit.

### Derivation 3.7: Depth, spectra, and the gradient histogram

**Setup.** For a plain deep network the backward recursion is $\delta_{k-1} = J_k^{\top}\delta_k$, so

$$
\Vert \delta_0 \Vert_2 \; \in \; \left[ \Vert \delta_L \Vert_2 \prod_{k}\sigma_{\min}(J_k), \; \Vert \delta_L \Vert_2 \prod_{k}\sigma_{\max}(J_k) \right]
$$

Taking logs converts the product into a sum, which is the useful form:

$$
\log\Vert\delta_0\Vert_2 \approx \log\Vert\delta_L\Vert_2 + \sum_{k=1}^{L}\log\sigma_k
$$

By the law of large numbers over layers, $\sum_k \log\sigma_k \approx L\,\mathbb{E}[\log\sigma]$: the gradient norm is **log-normal with mean linear in depth**. Vanishing and exploding are the two signs of $\mathbb{E}[\log\sigma]$, and neither is a rare event — they are the generic behaviour unless $\mathbb{E}[\log\sigma] = 0$ is engineered.

**Format thresholds.** Underflow to zero occurs when $\Vert\delta_0\Vert$ falls below the smallest subnormal:

$$
L^{*} = \frac{\log(\text{min subnormal})}{\mathbb{E}[\log\sigma]}
$$

With $\mathbb{E}[\log\sigma] = \log 0.9 = -0.105$: fp16 ($\log 5.96\times10^{-8} = -16.6$) dies at $L^{*} \approx 158$ layers; fp32 ($\log 1.4\times10^{-45} = -103$) at $L^{*} \approx 980$. Loss scaling by $S = 2^{16}$ adds $\log 2^{16} = 11.1$ to the numerator, extending fp16's reach to $L^{*} \approx 264$ — **loss scaling buys depth, measured in layers.**

**Residual connections, spectrally.** With $J_k = I + \tilde{J}_k$ and $\Vert\tilde{J}_k\Vert_2 = \beta \ll 1$:

$$
\sigma_{\max}(J_k) \le 1 + \beta, \qquad \sigma_{\min}(J_k) \ge 1 - \beta \quad \Longrightarrow \quad \prod_k \sigma \in \left[ (1-\beta)^{L}, (1+\beta)^{L} \right]
$$

For $\beta L = O(1)$ — which is what $1/\sqrt{L}$-scaled residual branches and zero-initialized final layers arrange — the product stays $\Theta(1)$ **independently of $L$**. This is the precise sense in which residual networks solve the numerical problem, not merely the optimization one.

**Clipping as a range guard.** $g \leftarrow g\min(1, c/\Vert g\Vert_2)$ enforces $\Vert g \Vert \le c$, guaranteeing the optimizer never sees a value near the format maximum. Note it does *not* prevent forward-pass overflow, and it changes the descent direction only in magnitude — the direction is preserved, which is why it is safe.

**Instrumentation.** The practical translation of this derivation: log $\log_{10}\Vert g_k\Vert$ per layer, per step. A straight line in depth is the log-normal prediction; its slope is $\mathbb{E}[\log\sigma]$, and comparing that slope against the format thresholds above tells you exactly how many layers you can afford before changing precision, scaling, or architecture.

## 4. Computational & Algorithmic Insights

### 4.1 The precision-allocation table

| Quantity | Recommended format | Reason |
|---|---|---|
| Matmul inputs (weights, activations) | bf16 / fp16 / fp8 | Bandwidth-bound; tolerant to $u = 10^{-2}$ noise (Topic 04) |
| Matmul accumulation | fp32 (in-hardware) | Reduction length $n \gg 1/u_{16}$; stalling otherwise |
| Master weights | fp32 (or fp32 + bf16 pair) | Updates are $10^{-5}$-relative, below $u_{16}/2$ |
| Optimizer state ($m$, $v$) | fp32, or 8-bit with block scaling | $v \sim g^{2}$ underflows; $\epsilon$ underflows |
| Loss and metric accumulators | fp32 / fp64 | Long reductions; absorption freezes metrics |
| Normalization statistics | fp32 | Cancellation amplification $\mu^{2}/\sigma^{2}$ |
| Softmax / log-sum-exp internals | fp32 | Exponent range; small denominators |
| Gradient all-reduce | bf16 / fp16 with fp32 accumulate | Bandwidth-bound; tree reduction depth $\log W$ |
| Attention scores before softmax | fp32 (or fp16 with $1/\sqrt{d_k}$ + max subtraction) | $\log(65504) = 11.09$ is small |
| Embedding / positional tables | bf16 fine | Lookups, no arithmetic |

**Reading the table.** Two columns of reasoning generate the whole thing: *reductions and cancellations go wide*; *streaming, elementwise, bandwidth-bound work goes narrow*. Everything else follows.

### 4.2 Debugging non-finite values

1. **Find the first non-finite tensor, not the last.** `NaN` propagates, so the loss is the last place to look. Use `torch.autograd.set_detect_anomaly(True)` (slow, but names the offending op) or register forward/backward hooks asserting `torch.isfinite(t).all()`.
2. **Distinguish overflow from `0/0`.** `inf` means a range failure (exponent); `NaN` from `inf - inf`, `0/0`, or `inf * 0` usually means a range failure *followed by* a cancellation. `NaN` in the gradient with a finite loss points at the backward formula (e.g. $\sqrt{x}$ at $x = 0$, $\log$ at $0$, `pow` with negative base).
3. **Bisect precision.** Rerun the failing step in fp32 (or fp64). If it succeeds, the bug is numerical, not logical, and the culprit is in the precision-allocation table above.
4. **Check the usual suspects, in order**: unfused softmax/log/sigmoid; variance or norm without $\epsilon$; division by a count that can be zero (masked tokens, empty batches); `sqrt` of a quantity that can be $0$ or slightly negative; `exp` of an unbounded score; an infinite value in the *data*; an fp16 $\epsilon$ that has flushed to zero.
5. **Guard rather than clamp.** `torch.nan_to_num` hides the bug and lets the run continue with corrupted state — the corruption then reaches the optimizer's momentum buffers and persists forever. Prefer to skip the step (as dynamic loss scaling does) and log the event.
6. **Watch the optimizer state.** A single `NaN` reaching $m$ or $v$ poisons that coordinate permanently, because $\mathrm{NaN} \cdot \beta + \dots = \mathrm{NaN}$. Check state finiteness after any anomaly, and be prepared to reset the affected entries.

### 4.3 Stable kernels you should never re-implement

| Task | Use | Never |
|---|---|---|
| $\log\sum e^{z}$ | `scipy.special.logsumexp`, `torch.logsumexp` | `log(sum(exp(z)))` |
| Softmax cross-entropy | `F.cross_entropy(logits, y)` | `log(softmax(z))[y]` |
| Binary cross-entropy | `F.binary_cross_entropy_with_logits` | `F.binary_cross_entropy(sigmoid(z), y)` |
| $\log(1+x)$, $e^{x}-1$ | `log1p`, `expm1` | `log(1+x)`, `exp(x)-1` |
| $\log\sigma(z)$ | `F.logsigmoid` | `log(sigmoid(z))` |
| $\sqrt{a^{2}+b^{2}}$ | `hypot` | `sqrt(a*a + b*b)` |
| Softmax over long sequences | fused/Flash attention kernels | manual score materialization |
| Variance / std | two-pass or Welford (framework default) | $\overline{x^{2}} - \bar{x}^{2}$ |
| $\log$ of a probability product | sum of log-probs | $\log$ of the product |
| Normalized weights for sampling | Gumbel-max on logits | normalize then sample |

Each right-hand column entry is a specific theorem from Topics 02–03 being violated. The left column is a decade of numerical analysis already compiled into a library call — the highest-return line of code in numerical deep learning is the one that deletes a hand-rolled softmax.

## 5. Real-World Physics & AI/ML Applications

### 5.1 Large-language-model pretraining

A contemporary pretraining run is a precision-allocation exercise executed at scale. Weights and activations are bf16 (chosen over fp16 precisely to avoid loss scaling at a scale where a divergence costs six figures); tensor cores accumulate in fp32; master weights and Adam moments are fp32, sharded across data-parallel ranks (ZeRO) because they dominate memory at $12$–$16$ bytes per parameter versus bf16's $2$; gradient all-reduce runs in bf16 with fp32 accumulation and a tree topology whose depth $\log_2 W$ (not $W$) enters the error bound (Topic 02).

fp8 training adds **per-tensor scaling factors**: each tensor is multiplied by a scale chosen from a running maximum (the "amax history") so its values fill E4M3's tiny $[2 \times 10^{-3}, 448]$ window, with the scale folded out afterwards — loss scaling, generalized from one global constant to one constant per tensor per step. E5M2 (more exponent, less mantissa) is used for gradients, whose dynamic range is wider; E4M3 for weights and activations, whose precision matters more. The whole design is Theorem 2.4 applied tensor-wise.

The loss curve itself must be watched numerically: spikes correlate with attention logits growing until $\exp$ saturates, which is why **QK-layernorm**, **logit soft-capping**, and $z$-loss (a penalty on $\log\sum_j e^{z_j}$) have become standard — all three are range-window interventions in disguise.

### 5.2 Attention at long context

Attention computes $\mathrm{softmax}(QK^{\top}/\sqrt{d_k})V$. Three numerical facts shape every implementation:

1. **The $1/\sqrt{d_k}$ is a variance argument, and a range argument.** With unit-variance entries, $q^{\top}k$ has standard deviation $\sqrt{d_k}$ — for $d_k = 128$, that is $\pm 11$ at one sigma, already at fp16's $\log(65504) = 11.09$. Dividing by $\sqrt{d_k}$ returns the scores to $O(1)$, keeping the softmax's inputs inside the window and its Jacobian well conditioned.
2. **Max-subtraction is mandatory, and it tiles.** The online-softmax recurrence — track a running max $m$ and running normalizer $\ell$, rescaling both by $e^{m_{\text{old}} - m_{\text{new}}}$ when a new block raises the maximum — computes the exact softmax in one streaming pass. Every exponent stays $\le 0$.
3. **Therefore stability and speed coincide.** FlashAttention uses exactly this recurrence to avoid materializing the $N \times N$ score matrix, giving the $\Theta(N^{2}d^{2}/M)$ traffic of Topic 04. The numerically necessary rescaling is what makes the tiling possible: a rare case where the stable algorithm is also the fast one.

Long-context inference adds a further wrinkle: KV caches stored in fp8 or int8 need per-block scales, and the *dequantization* must happen before the softmax, not after, or the max-subtraction operates on the wrong grid.

### 5.3 Log-space arithmetic in probabilistic and physical models

Whenever a model multiplies many probabilities — HMM forward–backward, particle filters, normalizing-flow log-likelihoods, importance sampling, sequence-level beam search, MCMC acceptance ratios — the product underflows within a few hundred terms in *any* format ($0.1^{308}$ ends fp64). The universal fix is to work in log space and replace multiplication by addition:

$$
\prod_i p_i \; \to \; \sum_i \log p_i, \qquad \sum_i p_i \; \to \; \operatorname{logsumexp}_i \log p_i
$$

The second replacement is Theorem 2.2, and it is the only nontrivial one: additions in probability space become log-sum-exps, each requiring a max-subtraction. Physics adds the same pattern with different names — partition functions $Z = \sum_s e^{-\beta E_s}$ are log-sum-exps over states, and the free energy $F = -\beta^{-1}\log Z$ is computed by subtracting the minimum energy first, exactly the max-subtraction with a sign flip.

In reinforcement learning, importance ratios $\pi_{\text{new}}/\pi_{\text{old}}$ are computed as $\exp(\log\pi_{\text{new}} - \log\pi_{\text{old}})$ for the same reason — the difference of log-probabilities is well conditioned while the ratio of probabilities is a quotient of two underflow-prone quantities. PPO's ratio clipping then doubles as a range guard.

### 5.4 Scientific machine learning: when fp32 is not enough

Physics-informed neural networks, neural ODEs, and learned PDE surrogates break the deep-learning consensus that low precision suffices, because their losses contain **derivative operators**, which are ill-conditioned by construction. A second-derivative term evaluated by finite differences has condition number $\Theta(h^{-2})$ and an optimal step $h^{*} \sim u^{1/4}$ with achievable accuracy $u^{1/2}$ (compare Topic 02's $u^{1/3}$/$u^{2/3}$ for first derivatives): in fp32 that caps the accuracy of a second derivative at $\approx 2 \times 10^{-4}$ — often larger than the residual being minimized.

Consequences seen in practice: PINN losses that plateau at a level set by precision rather than by optimization; stiff neural ODEs whose adaptive solvers reject every step because the local error estimate is dominated by rounding; and long-horizon rollouts whose energy drift is a floating-point bias, not a modelling error (Topic 02, Sec. 5.3). The standard responses are to compute derivatives by **automatic differentiation** rather than finite differences (exact to $O(u)$, no $h$ to tune), to keep the residual computation in fp64 while the network runs in fp32, and to non-dimensionalize the problem so that all quantities are $O(1)$ — a rescaling that is simultaneously a conditioning fix (Topic 03, Sec. 4.2) and a range-window fix.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Location |
|---|---|---|
| Format parameters, subnormals, FTZ | IEEE 754-2019; Muller et al., *Handbook of Floating-Point Arithmetic* (2018) | Ch. 3 |
| Log-sum-exp trick and its error analysis | Blanchard, Higham & Mary, *Accurate computation of the log-sum-exp and softmax functions* (IMA J. Numer. Anal., 2021) | Sec. 2–3 |
| Fused cross-entropy from logits | Goodfellow, Bengio & Courville, *Deep Learning* (2016) | Sec. 4.1, 6.2.2 |
| Mixed precision, loss scaling, master weights | Micikevicius et al., *Mixed Precision Training* (ICLR 2018) | Sec. 3.1–3.3 |
| fp8 formats and per-tensor scaling | Micikevicius et al., *FP8 Formats for Deep Learning* (2022) | Sec. 2–4 |
| bf16 rationale and empirical study | Kalamkar et al. (2019) | Sec. 3 |
| Rigorous mixed-precision theory | Higham & Mary, *Mixed precision algorithms in numerical linear algebra*, Acta Numerica 31 (2022) | Sec. 2, 5 |
| Summation/accumulator bounds | Higham (2002), Ch. 4; Blanchard, Higham & Mary (2020) | — |
| BatchNorm / LayerNorm / RMSNorm | Ioffe & Szegedy (2015); Ba et al. (2016); Zhang & Sennrich (2019) | — |
| Online variance for streaming statistics | Welford (1962); Chan, Golub & LeVeque (1983) | — |
| Adam and the role of $\epsilon$ | Kingma & Ba (ICLR 2015); Reddi, Kale & Kumar (ICLR 2018) | Alg. 1; Sec. 3 |
| Vanishing/exploding gradients, clipping | Pascanu, Mikolov & Bengio (ICML 2013) | Sec. 2–3 |
| Variance-preserving initialization | Glorot & Bengio (AISTATS 2010); He et al. (ICCV 2015) | — |
| Online softmax and tiled attention | Milakov & Gimelshein (2018); Dao et al., *FlashAttention* (NeurIPS 2022) | Sec. 3 |
| Conditioning framework used throughout | Trefethen & Bau (1997); Higham (2002) | Lectures 12–15; Ch. 7 |

**Module complete.** The five topics form one argument: [Topic 01](../01_ieee754_floating_point_representation/README.md) defines the arithmetic, [Topic 02](../02_error_propagation_and_stability_tricks/README.md) tracks how its errors compose, [Topic 03](../03_conditioning_and_condition_numbers/README.md) separates the problem's fault from the algorithm's, [Topic 04](../04_vectorization_and_numpy_performance/README.md) prices the data movement, and this topic spends all four budgets at once — in the narrowest arithmetic anyone has ever trusted with a billion parameters.